# TMJDLC – Site Access Tester

Batch-test candidate pages from Google Colab with Requests and Playwright Chromium. Records HTTP status, redirects, page title, visible text, and likely challenge/block status.

Diagnostic only: does not solve CAPTCHAs or bypass access controls. Run conservatively; sites may rate-limit repeated requests.

In [ ]:
!pip -q install requests beautifulsoup4 pandas playwright
!playwright install chromium

## 1. Configure URLs
Edit this list to add or remove candidates.

In [ ]:
URLS = [
    # URLs supplied for testing
    "https://javct.net/v/mida-838-rm",
    "https://www.javdock.com/v8/video/dasd-491/",
    "https://jjavhd.net/uncensored-leaked-jur-897/",
    "https://javhd.free/mida-841/",
    "https://supjav.com/468225.html",

    # Additional candidate homepages; not yet tested/verified
    "https://javhdporn.net/",
    "https://njav.tv/",
    "https://xsz-av.com/",
    "https://javtrailers.com/",
    "https://24-jav.com/",
    "https://javhd.today/",
    "https://javwow.com/",
    "https://javtiful.com/",
]

REQUEST_TIMEOUT = 20
BROWSER_TIMEOUT_MS = 45000
DELAY_BETWEEN_SITES_SECONDS = 2
MAX_TEXT_CHARS = 2500
print(f"{len(URLS)} URLs queued")

## 2. Run Requests and browser checks
Checks run sequentially. Challenges are reported, not bypassed.

In [ ]:
import asyncio, re, json
from datetime import datetime, timezone
from urllib.parse import urlparse
import requests
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

CHALLENGE_MARKERS = [
    "just a moment", "checking your browser", "verify you are human",
    "security check", "attention required", "enable javascript and cookies"
]

def classify(status, title="", body=""):
    text = f"{title} {body}".lower()
    if any(marker in text for marker in CHALLENGE_MARKERS):
        return "CHALLENGE"
    if status in (401, 403, 429):
        return "BLOCKED"
    if status is not None and 200 <= status < 300:
        return "ACCESSIBLE"
    if status is not None and 300 <= status < 400:
        return "REDIRECT"
    return "FAILED"

def check_requests(url):
    out = {"requests_status": None, "requests_final_url": "", "requests_title": "",
           "requests_content_type": "", "requests_result": "FAILED", "requests_error": ""}
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True,
                         headers={"User-Agent": "Mozilla/5.0"})
        out["requests_status"] = r.status_code
        out["requests_final_url"] = r.url
        out["requests_content_type"] = r.headers.get("content-type", "")
        if "html" in out["requests_content_type"].lower():
            soup = BeautifulSoup(r.text[:1_000_000], "html.parser")
            out["requests_title"] = soup.title.get_text(" ", strip=True) if soup.title else ""
        out["requests_result"] = classify(r.status_code, out["requests_title"], r.text[:MAX_TEXT_CHARS])
    except Exception as e:
        out["requests_error"] = f"{type(e).__name__}: {str(e)[:250]}"
    return out

async def check_browser(browser, url):
    out = {"browser_status": None, "browser_final_url": "", "browser_title": "",
           "browser_result": "FAILED", "browser_error": "", "visible_text_sample": ""}
    page = await browser.new_page()
    try:
        response = await page.goto(url, wait_until="domcontentloaded", timeout=BROWSER_TIMEOUT_MS)
        out["browser_status"] = response.status if response else None
        out["browser_final_url"] = page.url
        out["browser_title"] = await page.title()
        try:
            body = await page.locator("body").inner_text(timeout=10000)
        except Exception:
            body = ""
        out["visible_text_sample"] = re.sub(r"\\s+", " ", body)[:MAX_TEXT_CHARS]
        out["browser_result"] = classify(out["browser_status"], out["browser_title"], out["visible_text_sample"])
    except Exception as e:
        out["browser_error"] = f"{type(e).__name__}: {str(e)[:250]}"
    finally:
        await page.close()
    return out

async def run_checks(urls):
    rows = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        for i, url in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] {url}")
            row = {"tested_at_utc": datetime.now(timezone.utc).isoformat(),
                   "url": url, "domain": urlparse(url).netloc}
            row.update(check_requests(url))
            row.update(await check_browser(browser, url))
            rows.append(row)
            print(f"  Requests: {row['requests_status']} ({row['requests_result']}) | "
                  f"Browser: {row['browser_status']} ({row['browser_result']}) | {row['browser_title'][:90]}")
            if i < len(urls):
                await asyncio.sleep(DELAY_BETWEEN_SITES_SECONDS)
        await browser.close()
    return rows

results = await run_checks(URLS)
df = pd.DataFrame(results)
display(df[["url", "requests_status", "requests_result", "browser_status", "browser_result", "browser_title"]])

## 3. Export results
Download CSV and JSON reports for review.

In [ ]:
from google.colab import files
csv_path = "/content/site_access_results.csv"
json_path = "/content/site_access_results.json"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
files.download(csv_path)
files.download(json_path)

## Result meanings

- **ACCESSIBLE** – successful HTTP response without obvious challenge markers; verify title and visible text.
- **CHALLENGE** – likely browser/security challenge page.
- **BLOCKED** – HTTP 401, 403, or 429.
- **REDIRECT** – redirect response.
- **FAILED** – timeout, DNS, connection, or other failure.

A 200 response alone does not prove the site is scrape-ready. Inspect the visible text sample and final URL. This notebook does not extract media streams.